# Part 2 Report: From MLMC (Giles 2008) to MLQMC (Giles 2009)

## 1. Introduction and objective

This notebook is an upgraded report-style implementation of the multilevel framework introduced by Giles (2008), together with a critical extension discussion based on Giles and Waterhouse (2009), where quasi-Monte Carlo is combined with multilevel simulation. The goal is not only to reproduce key numerical patterns, but also to explain why they arise, what assumptions support them, where the method is strongest, and where limitations remain.

The computational problem is to estimate option values of the form $\mathbb{E}[P]$ when the underlying asset follows an SDE and the payoff may depend on the full path. In a standard Monte Carlo approach, decreasing discretisation bias requires smaller timesteps, but this increases per-path cost; simultaneously reducing statistical error requires more paths. This coupling of bias and variance is what drives the classical $O(\varepsilon^{-3})$ complexity for first-order weak schemes.

Giles (2008) resolves this tension by distributing work across a hierarchy of timestep levels and estimating only corrections between adjacent levels. This notebook implements that core mechanism for a European call under GBM with Euler timestepping and refinement factor $M=4$, matching the canonical setup in the paper. The report then interprets the empirical results and places them in context with the 2009 quasi-Monte Carlo extension.

## 2. Core method and experimental setup

The multilevel estimator is based on the telescoping identity
$$
\mathbb{E}[\hat{P}_L] = \mathbb{E}[\hat{P}_0] + \sum_{l=1}^{L}\mathbb{E}[\hat{P}_l-\hat{P}_{l-1}],
$$
where $\hat{P}_l$ denotes the payoff computed on timestep $h_l=M^{-l}T$. The key variance reduction step is coupling: on level $l$, fine and coarse payoffs are generated from the same Brownian path, with coarse increments obtained by summing blocks of fine increments. Because the two discretisations are close pathwise, the correction $\hat{P}_l-\hat{P}_{l-1}$ has much smaller variance than $\hat{P}_l$ itself.

The implementation below follows this logic directly: it estimates per-level variances from pilot samples, allocates samples approximately proportionally to $\sqrt{V_l h_l}$, and increases the finest level until a practical bias criterion is met. The numerical sections then study three properties expected from theory: decay of correction variance, decay of level means (bias proxy), and scaling of computational cost with target accuracy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(123)

# Parameters (European call, GBM — match Giles 2008 §6.1.1)
r_mlmc = 0.05
sigma_mlmc = 0.2
T_mlmc = 1.0
S0_mlmc = 1.0
K_mlmc = 1.0
M = 4  # refinement factor (paper uses M=4)

In [ ]:
def euler_gbm_path(S0, r, sigma, T, dW):
    """Single Euler path: S_{n+1} = S_n + r*S_n*h + sigma*S_n*dW_n. dW shape (n_steps,). Returns S_T."""
    n = len(dW)
    h = T / n
    S = S0
    for i in range(n):
        S = S + r * S * h + sigma * S * dW[i]
    return S

def euler_gbm_path_full(S0, r, sigma, T, dW):
    """Euler path returning full trajectory (for Asian etc). dW shape (n_steps,)."""
    n = len(dW)
    h = T / n
    S = np.zeros(n + 1)
    S[0] = S0
    for i in range(n):
        S[i + 1] = S[i] + r * S[i] * h + sigma * S[i] * dW[i]
    return S

def payoff_european(S_T, K, r, T):
    """Discounted European call payoff."""
    return np.exp(-r * T) * np.maximum(S_T - K, 0.0)

In [ ]:
def mlmc_coupled_paths(l, S0, r, sigma, T, K, M, seed=None):
    """
    One coupled (fine, coarse) path at level l.
    Fine: timestep h_l = T/M^l, coarse: h_{l-1} = T/M^{l-1}.
    Same Brownian path: coarse increments = sum of M fine increments.
    Returns (P_l, P_{l-1}) for European call.
    """
    if seed is not None:
        np.random.seed(seed)
    n_fine = M ** l
    h_fine = T / n_fine
    dW_fine = np.random.standard_normal(n_fine) * np.sqrt(h_fine)
    S_fine = euler_gbm_path(S0, r, sigma, T, dW_fine)
    if l == 0:
        return payoff_european(S_fine, K, r, T), 0.0  # level 0: no coarse path
    # Coarse: sum fine increments in groups of M
    dW_coarse = dW_fine.reshape(-1, M).sum(axis=1)
    S_coarse = euler_gbm_path(S0, r, sigma, T, dW_coarse)
    P_fine = payoff_european(S_fine, K, r, T)
    P_coarse = payoff_european(S_coarse, K, r, T)
    return P_fine, P_coarse

def mlmc_level_samples(l, N, S0, r, sigma, T, K, M, seed=None):
    """N coupled samples at level l. Returns (diffs, mean_diff, fine_payoffs)."""
    if seed is not None:
        np.random.seed(seed)
    if l == 0:
        out = np.array([mlmc_coupled_paths(0, S0, r, sigma, T, K, M)[0] for _ in range(N)])
        return out, np.mean(out), out  # P_0, mean, and "fine" = P_0
    diffs = np.zeros(N)
    fine_payoffs = np.zeros(N)
    for i in range(N):
        Pf, Pc = mlmc_coupled_paths(l, S0, r, sigma, T, K, M)
        diffs[i] = Pf - Pc
        fine_payoffs[i] = Pf
    return diffs, np.mean(diffs), fine_payoffs

In [ ]:
def run_mlmc(eps, S0, r, sigma, T, K, M, N_pilot=2000):
    """
    MLMC algorithm (Giles §5). Target RMSE epsilon; cost = total timesteps.
    Returns: (L, N_l list, Y_hat, cost, V_est, Y_est).
    """
    L = 0
    V_est, Y_est = {}, {}
    while True:
        # Pilot at new level L
        if L == 0:
            s0, _, _ = mlmc_level_samples(0, N_pilot, S0, r, sigma, T, K, M)
            V_est[0] = np.var(s0)
            Y_est[0] = np.mean(s0)
        else:
            sL, _, _ = mlmc_level_samples(L, N_pilot, S0, r, sigma, T, K, M)
            V_est[L] = np.var(sL)
            Y_est[L] = np.mean(sL)
        # Optimal N_l (eq 12)
        sum_sqrt = sum(np.sqrt(V_est[l] * (T / M**l)) for l in range(L + 1))
        sum_sqrt = max(sum_sqrt, 1e-20)
        N_opt = []
        for l in range(L + 1):
            hl = T / (M ** l)
            n_l = int(np.ceil(2 * (eps**-2) * np.sqrt(V_est[l] * hl) / sum_sqrt))
            N_opt.append(max(n_l, 100))
        # Run N_opt[l] samples at each level and recompute mean/var
        for l in range(L + 1):
            if l == 0:
                s0, _, _ = mlmc_level_samples(0, N_opt[0], S0, r, sigma, T, K, M)
                Y_est[0], V_est[0] = np.mean(s0), np.var(s0)
            else:
                sl, _, _ = mlmc_level_samples(l, N_opt[l], S0, r, sigma, T, K, M)
                Y_est[l], V_est[l] = np.mean(sl), np.var(sl)
        cost = N_opt[0] * 1 + sum(N_opt[l] * (M**l + M**(l-1)) for l in range(1, L + 1))
        # Bias check (eq 10)
        thresh = (1 / np.sqrt(2)) * (M - 1) * eps
        if L >= 2:
            if max(abs(Y_est[L]) * 1.0, abs(Y_est.get(L - 1, 0)) / M) < thresh:
                break
        if L >= 5:
            break
        L += 1
    Y_hat = sum(Y_est[l] for l in range(L + 1))
    return L, N_opt, Y_hat, cost, V_est, Y_est

In [ ]:
# Single MLMC run at target accuracy epsilon
eps_target = 0.001
L, N_list, Y_hat, cost, V_est, Y_est = run_mlmc(
    eps_target, S0_mlmc, r_mlmc, sigma_mlmc, T_mlmc, K_mlmc, M, N_pilot=2000
)
print(f"Target epsilon = {eps_target}")
print(f"Levels L = 0..{L}, N_l = {N_list}")
print(f"MLMC estimate Y_hat = {Y_hat:.6f}")
print(f"Total cost (timesteps) = {cost:.0f}")
print(f"V_l (sample var of P_l - P_{{l-1}}) = {[round(V_est[l], 8) for l in range(L+1)]}")

In [ ]:
# Variance decay: V_l and E[P_l - P_{l-1}] vs level (reproducing paper Figure 2 left)
np.random.seed(456)
n_per_level = 5000
max_L = 6
vars_Pl = []
vars_diff = []
means_diff = []
for l in range(max_L + 1):
    out, m, fine = mlmc_level_samples(l, n_per_level, S0_mlmc, r_mlmc, sigma_mlmc, T_mlmc, K_mlmc, M)
    vars_Pl.append(np.var(fine))
    vars_diff.append(np.var(out))
    means_diff.append(m)
levels = np.arange(max_L + 1)
hl = T_mlmc / (M ** levels)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(levels, np.log(np.array(vars_Pl) + 1e-20) / np.log(M), 'o-', label=r'$\log_M \mathrm{Var}(\hat{P}_l)$')
ax[0].plot(levels, np.log(np.array(vars_diff) + 1e-20) / np.log(M), 's-', label=r'$\log_M \mathrm{Var}(\hat{P}_l - \hat{P}_{l-1})$')
ax[0].axhline(-1, color='gray', linestyle='--', alpha=0.7, label='slope -1 (O(h))')
ax[0].set_xlabel('Level l')
ax[0].set_ylabel(r'$\log_M$ variance')
ax[0].legend()
ax[0].set_title('Variance decay (European call, GBM)')
ax[1].semilogy(levels, np.array(np.abs(means_diff)) + 1e-20, 'o-', label=r'$|\mathbb{E}[\hat{P}_l - \hat{P}_{l-1}]|$')
ax[1].semilogy(levels, hl, '--', label=r'$h_l$')
ax[1].set_xlabel('Level l')
ax[1].set_ylabel('|mean| or h_l')
ax[1].legend()
ax[1].set_title('Bias decay')
plt.tight_layout()
plt.show()

In [ ]:
# Cost comparison: epsilon^2 * cost vs epsilon (MLMC vs standard MC), as in paper Figure 2
def standard_mc_cost(eps, S0, r, sigma, T, K, M):
    """Standard MC: N paths with L steps so bias O(h_L) ~ eps. Cost = N * M^L."""
    L = max(0, int(np.ceil(np.log(T / eps) / np.log(M))))
    n_steps = M ** L
    var_P = 0.02  # approximate
    N = int(np.ceil(2 * var_P * (eps ** -2)))
    return N * n_steps, L

eps_list = [0.005, 0.002, 0.001, 0.0005]
costs_mlmc = []
costs_std = []
estimates_mlmc = []
for eps in eps_list:
    L, N_list, Yh, cost, _, _ = run_mlmc(eps, S0_mlmc, r_mlmc, sigma_mlmc, T_mlmc, K_mlmc, M, N_pilot=1500)
    costs_mlmc.append(cost)
    estimates_mlmc.append(Yh)
    c_std, _ = standard_mc_cost(eps, S0_mlmc, r_mlmc, sigma_mlmc, T_mlmc, K_mlmc, M)
    costs_std.append(c_std)

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.loglog(eps_list, np.array(eps_list)**2 * np.array(costs_mlmc), 'o-', label='MLMC')
ax.loglog(eps_list, np.array(eps_list)**2 * np.array(costs_std), 's--', label='Standard MC')
ax.set_xlabel(r'$\varepsilon$')
ax.set_ylabel(r'$\varepsilon^2 \times$ cost')
ax.legend()
ax.set_title(r'Computational cost: MLMC vs standard MC (European call)')
plt.tight_layout()
plt.show()
print('MLMC epsilon^2 * cost:', [round(eps**2 * c, 2) for eps, c in zip(eps_list, costs_mlmc)])
print('Std MC epsilon^2 * cost:', [round(eps**2 * c, 2) for eps, c in zip(eps_list, costs_std)])

In [ ]:
# Reference value: Black–Scholes European call (for comparison)
from scipy.stats import norm
def bs_european_call(S0, K, r, sigma, T):
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return np.exp(-r*T) * (S0*norm.cdf(d1) - K*norm.cdf(d2))
ref_value = bs_european_call(S0_mlmc, K_mlmc, r_mlmc, sigma_mlmc, T_mlmc)
print(f"Black–Scholes European call value = {ref_value:.6f}")
print(f"MLMC estimate at eps={eps_target} was Y_hat = {Y_hat:.6f} (error {abs(Y_hat - ref_value):.6f})")

## 3. Main results from Giles (2008): interpretation of this notebook output

The first substantive finding is that coupling works exactly as intended: the variance of $\hat{P}_l-\hat{P}_{l-1}$ declines rapidly with level, approximately in proportion to $h_l$ in this Euler/European-call setting. This is the mechanism that makes multilevel efficient. If one estimated only $\mathbb{E}[\hat{P}_L]$ on the finest grid, variance would stay comparatively large while each sample remains expensive; by estimating small corrections instead, one can reserve most expensive fine-grid work for a small fraction of total samples.

The second finding is that level means $|\mathbb{E}[\hat{P}_l-\hat{P}_{l-1}]|$ decrease with refinement in a way consistent with first-order weak convergence. Practically, this gives a data-driven stopping rule for selecting the finest level: once the last correction terms are sufficiently small, adding levels gives negligible bias reduction relative to cost.

The third finding concerns complexity. Standard Monte Carlo with first-order weak discretisation needs roughly $O(\varepsilon^{-2})$ paths and $O(\varepsilon^{-1})$ timesteps per path to achieve RMS error $\varepsilon$, yielding $O(\varepsilon^{-3})$ cost. In contrast, MLMC concentrates computational effort on cheap coarse levels and only lightly samples expensive fine levels. This is why the empirical $\varepsilon^2\times\text{cost}$ curve for MLMC grows much more slowly than for standard MC.

## 4. Strengths and weaknesses of the 2008 method

A central strength is robustness: MLMC does not depend on highly specialized payoff smoothness beyond what is needed to control weak/strong errors of the discretisation. It also scales naturally to path-dependent options because the hierarchy and coupling strategy are model-agnostic once a consistent coarse-fine coupling is available. Another strength is practical transparency: levelwise variances and means are observable during runtime, allowing adaptive budget allocation and explicit diagnostics.

The main weakness is that the sampling engine on each level is still standard Monte Carlo. Therefore, once discretisation effects are controlled, the integration error in sample size remains fundamentally Monte Carlo-like, with $N^{-1/2}$ behavior. In high-accuracy regimes this becomes expensive. A related practical limitation is that benchmark studies are usually performed on synthetic models (for example GBM), so empirical gains in market-calibrated, data-driven pipelines still require separate validation.

## 5. Why Giles (2009) is an upgrade: multilevel quasi-Monte Carlo

Giles and Waterhouse (2009) keep the multilevel decomposition but replace i.i.d. sampling on each level with randomized lattice-rule quasi-Monte Carlo (QMC), using Brownian bridge construction to improve effective dimension. The conceptual upgrade is straightforward: MLMC reduces variance through hierarchy and coupling, while QMC improves integration efficiency for sufficiently smooth integrands. Combining both mechanisms typically lowers cost more than either method alone.

This addresses the sampling-inefficiency concern from the 2008 framework. In favorable settings (notably smooth payoffs such as European and Asian options), the QMC component can approach near-$N^{-1}$ behavior, which is substantially faster than Monte Carlo. The 2009 numerical evidence reports strong additional savings in $\varepsilon^2\times\text{cost}$ when compared with standard QMC and with MLMC alone.

However, the upgrade is not universal. QMC performance is sensitive to non-smooth structure in the integrand. Discontinuous or kink-dominated payoffs (for example digital/barrier-type features) can reduce the effective benefit, because the smoothness assumptions behind strong QMC rates are weakened. In those cases, MLMC remains useful, but the marginal gain from adding QMC may be smaller and more problem-dependent.

## 6. Comparative conclusion on exotic options and method choice

For exotic options with path dependence but relatively smooth functionals (such as Asian payoffs), the strongest practical recommendation is the combined MLQMC approach from 2009: multilevel handles time-discretisation economics, and QMC improves sampling efficiency on each level. For exotics with significant non-smoothness (digital barriers, sharp indicators), the safer baseline is often MLMC from 2008, with MLQMC tested empirically rather than assumed to dominate.

The broader methodological conclusion is that Giles (2008) provides the foundational architecture for complexity reduction, while Giles (2009) provides the integration upgrade that targets the remaining bottleneck. Together they form a coherent progression: first remove unnecessary fine-grid sampling through multilevel coupling, then accelerate remaining integration through low-discrepancy sampling where payoff regularity permits.

### References

Giles, M. B. (2008). Multilevel Monte Carlo path simulation. *Operations Research*, 56(3), 607-617.

Giles, M. B., and Waterhouse, B. J. (2009). Multilevel quasi-Monte Carlo path simulation. *Radon Series on Computational and Applied Mathematics*, 8, 1-18.